# Fit a posed skeleton to 2D keypoints — SMPLify mechanics

**Track A · Human Modeling** · maps to lessons A4 (HMR) and A9 (SMPLify).

SMPLify recovers a 3D body by *optimizing* its pose so that the projected joints match detected 2D keypoints, regularized by a pose prior. Here we use a small articulated skeleton with forward kinematics so the notebook is self-contained — the optimization loop is exactly the one used for real SMPL. The last section shows how to swap in the real SMPL body with `smplx`.

> CPU is fine.

In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 400))

## 1 · A tiny articulated skeleton + forward kinematics
A root plus a kinematic tree (spine, two arms, two legs). Each joint has a rotation about z; FK gives 3D joint positions, then a weak-perspective camera projects to 2D.

In [ ]:
# parent of each joint (-1 = root) and the bone offset from its parent
parents = [-1, 0, 1, 2, 1, 4, 1, 6, 0, 8, 0, 10]
offsets = torch.tensor([
    [0,0,0],[0,.5,0],[0,.4,0],[0,.35,0],[-.3,0,0],[-.35,0,0],
    [.3,0,0],[.35,0,0],[-.15,-.1,0],[0,-.7,0],[.15,-.1,0],[0,-.7,0]], dtype=torch.float32)

def fk(angles, root):
    R = lambda a: torch.stack([torch.stack([torch.cos(a),-torch.sin(a),torch.zeros_like(a)]),
                               torch.stack([torch.sin(a), torch.cos(a),torch.zeros_like(a)]),
                               torch.stack([torch.zeros_like(a),torch.zeros_like(a),torch.ones_like(a)])])
    glob = [None]*len(parents); pos = [None]*len(parents)
    for j,p in enumerate(parents):
        Rj = R(angles[j])
        if p==-1: glob[j]=Rj; pos[j]=root
        else:     glob[j]=glob[p]@Rj; pos[j]=pos[p]+ (glob[p]@offsets[j])
    return torch.stack(pos)

def project(P):                       # weak-perspective: drop z, scale
    return P[:, :2] * 1.0

## 2 · Make a target observation (the "detected" 2D keypoints)
Pose the skeleton with a known pose, project, and add noise — this stands in for an off-the-shelf 2D keypoint detector.

In [ ]:
true_ang = torch.zeros(len(parents)); true_ang[4]=-0.9; true_ang[6]=0.9; true_ang[2]=0.3; true_ang[9]=0.2
true_root = torch.tensor([0.,0.,0.])
target2d = project(fk(true_ang, true_root)) + 0.01*torch.randn(len(parents),2)
target2d = target2d.to(device)

## 3 · Optimize pose to match — reprojection loss + pose prior
Start from a rest pose and let Adam recover the angles. The prior keeps the solution plausible (small deviations), exactly as in SMPLify.

In [ ]:
ang  = torch.zeros(len(parents), device=device, requires_grad=True)
root = torch.zeros(3, device=device, requires_grad=True)
opt  = torch.optim.Adam([ang, root], 0.05)
off  = offsets.to(device)
hist = []
for step in range(STEPS + 1):
    pred2d = project(fk(ang, root))
    reproj = ((pred2d - target2d) ** 2).mean()
    prior  = 0.001 * (ang ** 2).mean()          # pose prior
    loss = reproj + prior
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        hist.append((step, reproj.item())); print(f"step {step:4d}  reproj {reproj.item():.5f}")

## 4 · Compare — before vs. fitted vs. target

In [ ]:
def draw(ax, P2, title, c):
    P2 = P2.detach().cpu()
    for j,p in enumerate(parents):
        if p!=-1: ax.plot([P2[j,0],P2[p,0]],[P2[j,1],P2[p,1]], c+"-", lw=2)
    ax.scatter(P2[:,0],P2[:,1], c=c, s=20); ax.set_title(title); ax.set_aspect("equal"); ax.axis("off")
fig, ax = plt.subplots(1, 3, figsize=(11, 4))
draw(ax[0], project(fk(torch.zeros(len(parents),device=device), torch.zeros(3,device=device))), "initial", "C7")
draw(ax[1], project(fk(ang, root)), "fitted", "C0")
ax[2].scatter(target2d.cpu()[:,0], target2d.cpu()[:,1], c="C3", s=20); ax[2].set_title("target 2D"); ax[2].set_aspect("equal"); ax[2].axis("off")
plt.tight_layout(); plt.show()

### Fitting the *real* SMPL body
The loop above is identical for SMPL — only the forward model changes. Install the parametric body and optimize `(β, θ, global)` against your 2D keypoints:

```python
!pip install smplx
import smplx
body = smplx.create("models", model_type="smpl")  # register + download at smpl-x.is.tue.mpg.de
out = body(betas=beta, body_pose=theta, global_orient=go)
joints2d = camera(out.joints)        # project, then minimize ||joints2d - keypoints|| + priors
```
For a learned single-shot regressor instead of per-image optimization, see **[4D-Humans / HMR 2.0](https://github.com/shubham-goel/4D-Humans)** and **[VIBE](https://github.com/mkocabas/VIBE)**.